# Hermes LoRA Worker (Colab)

Hermes (Keina) から投入された LoRA 学習ジョブを、この Colab の GPU で実行するワーカーです。

- ジョブ置き場: Drive の `ChatGPT-Obsidian-Vault/06_Automation/Hermes/LoraJobs/`
- 進め方: セルを上から順に実行してください。最後の「ワーカー本体」セルがジョブを待ち受けて処理します(既定で最大10時間)。
- 学習には kohya sd-scripts を使います。成果物は `.safetensors` としてジョブフォルダへ書き戻されます。
- このノートブックは同時に1つだけ動かしてください(二重に動かすとジョブを取り合います)。

- **落ちても大丈夫な設計です**: 学習中は500ステップごとのチェックポイントがその場でDriveへ送られ、セッションが切れても次回起動時にワーカーが自動で検出して、最後のチェックポイントから続きを学習します(最大でも500ステップ分の巻き戻し)。

**修正版（2026-11-07）**: ZIPをサブフォルダごと検証して利用、実在するモデル重みを検証し、矛盾するキャッシュ/学習オプションを修正。既存のノートブックと同時に起動しないでください。


In [ ]:
# 1) Google認証 + Driveマウント(標準手順。認証はご本人がColab画面で承認してください)
from google.colab import auth
auth.authenticate_user()
from google.colab import drive
drive.mount('/content/drive')
print('mounted')


In [ ]:
# 2) Vault と LoraJobs の場所を特定する
import os
from pathlib import Path

VAULT_NAME = 'ChatGPT-Obsidian-Vault'
candidates = []
for root in [Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives')]:
    if root.exists():
        for p in root.rglob(VAULT_NAME):
            candidates.append(p)
            break
if not candidates:
    raise RuntimeError('Vault folder not found. Driveに ChatGPT-Obsidian-Vault があるか確認してください(共有の場合は「マイドライブに追加」のショートカットが必要です)')
VAULT = candidates[0]
BASE = VAULT / '06_Automation' / 'Hermes' / 'LoraJobs'
BASE.mkdir(parents=True, exist_ok=True)
print('VAULT:', VAULT)
print('BASE :', BASE)


In [ ]:
# 3) kohya sd-scripts のセットアップ(初回のみ数分)
import subprocess, sys
from pathlib import Path

SD = Path('/content/sd-scripts')
if not SD.exists():
    subprocess.run(['git', 'clone', 'https://github.com/kohya-ss/sd-scripts.git', str(SD)], check=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SD / 'requirements.txt')], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


In [ ]:
# 4) ワーカー本体: ジョブを待ち受けて学習し、結果を書き戻す
import hashlib, json, os, re, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

POLL_SECONDS = 60
POLL_HOURS = 10  # Colabセッション上限に合わせて調整可

def now():
    return datetime.now(timezone.utc).isoformat()

def wjson(path, obj):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2))
    tmp.replace(path)

def rjson(path):
    try:
        return json.loads(path.read_text())
    except Exception:
        return None

def heartbeat(state='alive', note=None):
    import torch
    wjson(BASE / 'worker_health.json', {
        'state': state, 'updated_at': now(), 'note': note,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

def resolve_base(spec, work):
    """Resolve a REAL SDXL checkpoint or a fully populated Diffusers directory.

    A .complete flag and model_index.json alone do not prove weights exist.
    kohya's SDXL trainer accepts genuine Diffusers directories directly;
    converting to a single fp16 file is not required.
    """
    from huggingface_hub import snapshot_download
    from safetensors import safe_open
    repo = spec['base_repo']
    slug = repo.replace('/', '__')
    drive_cache = BASE / 'base_cache' / slug

    def valid_checkpoint(p):
        try:
            if not p.is_file() or p.stat().st_size < 1_000_000_000:
                return False
            with safe_open(str(p), framework='pt', device='cpu') as f:
                names = list(f.keys())
                has_unet = any(k.startswith('model.diffusion_model.') for k in names)
                has_text = any(k.startswith('conditioner.') for k in names)
                has_vae = any(k.startswith('first_stage_model.') for k in names)
                sample = names[:100]
                half = any(f.get_slice(k).get_dtype() in ('F16', 'BF16') for k in sample)
                return len(names) > 1000 and has_unet and has_text and has_vae and half
        except Exception as e:
            print('invalid single-file cache:', p, repr(e))
            return False

    if drive_cache.is_dir():
        for p in sorted(drive_cache.rglob('*.safetensors')):
            if valid_checkpoint(p):
                print('validated SDXL fp16 cache:', p)
                return str(p)

    def valid_diffusers(p):
        if not (p / 'model_index.json').is_file():
            return False
        for sub in ('unet', 'vae', 'text_encoder', 'text_encoder_2'):
            d = p / sub
            if not d.is_dir():
                return False
            if not any(f.is_file() and f.stat().st_size > 1_000_000
                       for f in list(d.glob('*.safetensors')) + list(d.glob('*.bin'))):
                return False
        return all((p / sub).is_dir() for sub in ('tokenizer', 'tokenizer_2'))

    if valid_diffusers(drive_cache):
        print('validated Diffusers Drive cache:', drive_cache)
        return str(drive_cache)

    print('Drive cache incomplete; downloading actual Diffusers weights to local disk:', repo)
    local = Path('/content/lora_base_models') / slug
    if not valid_diffusers(local):
        snapshot_download(repo_id=repo, local_dir=str(local), max_workers=4)
    if not valid_diffusers(local):
        raise RuntimeError(f'incomplete model download: {local}')
    print('validated local Diffusers weights:', local)
    return str(local)

def train_one(job_dir, job):
    spec = job['spec']
    job_id = job['job_id']
    status_p = job_dir / 'status.json'
    ckpt_p = job_dir / 'checkpoint.json'
    result_p = job_dir / 'result.json'
    wjson(status_p, {'job_id': job_id, 'state': 'running', 'updated_at': now()})
    work = Path('/content/lora_work') / job_id
    if work.exists():
        shutil.rmtree(work)
    (work / 'data').mkdir(parents=True)
    # The source ZIP has two nested concept folders.
    import zipfile
    ds_root = work / 'data'
    zip_path = job_dir / 'dataset.zip'
    if zip_path.is_file():
        with zipfile.ZipFile(zip_path) as z:
            bad = z.testzip()
            if bad:
                raise RuntimeError(f'corrupt ZIP member: {bad}')
            for info in z.infolist():
                parts = Path(info.filename).parts
                if info.filename.startswith('/') or '..' in parts:
                    raise RuntimeError(f'unsafe ZIP path: {info.filename}')
                z.extract(info, ds_root)
    else:
        src = job_dir / 'dataset'
        if not src.exists():
            raise RuntimeError('neither dataset.zip nor dataset/ exists')
        shutil.copytree(src, ds_root, dirs_exist_ok=True)
    img_exts = ('.png', '.jpg', '.jpeg', '.webp')
    subsets = spec.get('subsets') or []
    expected_total = int(spec.get('image_count', 0))
    actual_total = 0
    for sub in subsets:
        folder = ds_root / sub['dir']
        images = [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in img_exts] if folder.is_dir() else []
        count = len(images)
        if count != int(sub['image_count']):
            raise RuntimeError(f'dataset mismatch: {sub["dir"]}: {count} images, expected {sub["image_count"]}')
        missing = [f.name for f in images if not f.with_suffix('.txt').is_file()]
        if missing:
            raise RuntimeError(f'missing captions in {sub["dir"]}: {missing[:5]}')
        actual_total += count
    if actual_total != expected_total or actual_total == 0:
        raise RuntimeError(f'dataset mismatch: {actual_total} images, expected {expected_total}')
    print('DATASET VERIFIED', actual_total, 'images', len(subsets), 'subsets')
    base_arg = resolve_base(spec, work)
    # Resume support: interval checkpoints are streamed into the job
    # folder (Drive) during training, so a dead Colab session loses at
    # most the steps since the last 500-step checkpoint. On a fresh
    # session the newest checkpoint becomes --network_weights and only
    # the remaining steps are trained (the LR schedule restarts).
    resume_from, steps_done = None, 0
    total_steps = spec['max_train_steps']
    for cand in sorted(job_dir.glob('*.safetensors')):
        if cand.name == 'output.safetensors':
            continue
        nums = re.findall(r'(\d{3,})', cand.stem)
        if nums and int(nums[-1]) > steps_done:
            steps_done, resume_from = int(nums[-1]), cand
    if resume_from is not None and steps_done >= total_steps:
        dest0 = job_dir / 'output.safetensors'
        shutil.copy(resume_from, dest0)
        data0 = dest0.read_bytes()
        wjson(result_p, {'job_id': job_id, 'state': 'succeeded',
                         'output': 'output.safetensors',
                         'source_name': resume_from.name, 'bytes': len(data0),
                         'sha256': hashlib.sha256(data0).hexdigest(),
                         'base': spec['base'], 'base_repo': spec['base_repo'],
                         'steps': total_steps, 'finished_at': now(),
                         'note': 'finalized from streamed checkpoint after session loss'})
        wjson(status_p, {'job_id': job_id, 'state': 'succeeded', 'updated_at': now()})
        shutil.rmtree(work, ignore_errors=True)
        print('FINALIZED-FROM-CKPT', job_id, resume_from.name)
        return
    steps_to_run = total_steps - steps_done
    if resume_from is not None:
        print(f'RESUME {job_id}: from {resume_from.name} (step {steps_done}/{total_steps}), training {steps_to_run} more steps')
        wjson(ckpt_p, {'job_id': job_id, 'state': 'resuming', 'step': steps_done,
                       'total_steps': total_steps, 'updated_at': now()})
    out_dir = work / 'out'
    out_dir.mkdir()
    cmd = [sys.executable, '-m', 'accelerate.commands.launch', '--num_cpu_threads_per_process', '4',
           str(SD / 'sdxl_train_network.py'),
           f'--pretrained_model_name_or_path={base_arg}',
           f'--train_data_dir={work / "data"}',
           '--network_module=networks.lora',
           f"--network_dim={spec['network_dim']}",
           f"--network_alpha={spec['network_alpha']}",
           f"--learning_rate={spec['learning_rate']}",
           f"--max_train_steps={steps_to_run}",
           f"--resolution={spec['resolution']},{spec['resolution']}",
           f'--output_dir={out_dir}',
           f"--output_name={spec['name']}",
           '--save_model_as=safetensors',
           '--mixed_precision=fp16',
           '--gradient_checkpointing',
           '--cache_latents',
           '--cache_text_encoder_outputs',
           '--network_train_unet_only',
           '--no_half_vae',
           '--train_batch_size=1',
           '--max_data_loader_n_workers=0',
           '--optimizer_type=adamw8bit',
        '--enable_bucket',
           '--lr_scheduler=cosine',
           f"--seed={spec.get('seed', 42)}",
           "--save_every_n_steps=500",
           '--caption_extension=.txt',
           # shuffle_caption cannot be combined with cached text outputs
           ]
    if resume_from is not None:
        cmd.append(f'--network_weights={resume_from}')
    log_path = work / 'train.log'
    t0 = time.time()
    with open(log_path, 'w') as logf:
        proc = subprocess.Popen(cmd, cwd=str(SD), stdout=logf, stderr=subprocess.STDOUT)
        while proc.poll() is None:
            time.sleep(30)
            tail = log_path.read_text(errors='replace')[-2000:] if log_path.exists() else ''
            m = re.findall(r'(\d+)\s*/\s*(\d+)\s*\[', tail)
            step = int(m[-1][0]) if m else None
            total = int(m[-1][1]) if m else spec['max_train_steps']
            wjson(ckpt_p, {'job_id': job_id, 'state': 'running',
                           'step': (steps_done + step) if step is not None else None,
                           'total_steps': total_steps, 'elapsed_sec': int(time.time() - t0),
                           'updated_at': now()})
            heartbeat('training', job_id)
            for o in sorted(out_dir.glob('*.safetensors')):
                dst = job_dir / o.name
                if not dst.exists():
                    try:
                        shutil.copy(o, dst)
                        print('CKPT->Drive', o.name)
                    except Exception as e:
                        print('ckpt sync failed (continuing):', e)
    tail = log_path.read_text(errors='replace')[-3000:] if log_path.exists() else ''
    if log_path.exists():
        shutil.copy(log_path, job_dir / 'train.log')
    if proc.returncode != 0:
        raise RuntimeError(f'kohya exited {proc.returncode}: {tail[-1500:]}')
    outs = sorted(out_dir.glob('*.safetensors'))
    if not outs:
        raise RuntimeError('no safetensors produced')
    final = outs[-1]
    for o in outs:
        if o != final:
            shutil.copy(o, job_dir / o.name)  # 500-step interval checkpoints
    dest = job_dir / 'output.safetensors'
    shutil.copy(final, dest)
    data = dest.read_bytes()
    result = {'job_id': job_id, 'state': 'succeeded', 'output': 'output.safetensors',
              'source_name': final.name, 'bytes': len(data),
              'sha256': hashlib.sha256(data).hexdigest(),
              'base': spec['base'], 'base_repo': spec['base_repo'],
              'steps': spec['max_train_steps'], 'finished_at': now()}
    wjson(result_p, result)
    wjson(status_p, {'job_id': job_id, 'state': 'succeeded', 'updated_at': now()})
    shutil.rmtree(work, ignore_errors=True)
    print('DONE', job_id, result['sha256'][:12], len(data))

REPAIR_RETRY_IDS = {'job-lora-24658ede902d4436'}
repair_attempted = set()
deadline = time.time() + POLL_HOURS * 3600
print('worker start; polling every', POLL_SECONDS, 's for', POLL_HOURS, 'h')
while time.time() < deadline:
    heartbeat('alive')
    try:
        for job_dir in sorted(BASE.iterdir()):
            if not job_dir.is_dir() or not job_dir.name.startswith('job-lora-'):
                continue
            status = rjson(job_dir / 'status.json') or {}
            job = rjson(job_dir / 'job.json')
            state = status.get('state')
            stale = False
            if state == 'running':
                ck = rjson(job_dir / 'checkpoint.json') or {}
                upd = ck.get('updated_at') or status.get('updated_at') or ''
                try:
                    stale = (datetime.now(timezone.utc) - datetime.fromisoformat(upd)).total_seconds() > 900
                except Exception:
                    stale = True
            retry_failed = (state == 'failed' and job_dir.name in REPAIR_RETRY_IDS
                            and job_dir.name not in repair_attempted)
            if job and job.get('spec', {}).get('type') == 'lora_train' and (state == 'queued' or stale or retry_failed):
                repair_attempted.add(job_dir.name)
                if stale:
                    print('RECOVER stale running job (previous session died):', job_dir.name)
                print('START', job_dir.name)
                try:
                    train_one(job_dir, job)
                except Exception as e:
                    wjson(job_dir / 'result.json', {'job_id': job_dir.name, 'state': 'failed',
                                                    'error': f'{type(e).__name__}: {e}', 'finished_at': now()})
                    wjson(job_dir / 'status.json', {'job_id': job_dir.name, 'state': 'failed', 'updated_at': now()})
                    print('FAILED', job_dir.name, e)
    except Exception as e:
        print('poll error (continuing):', e)
    time.sleep(POLL_SECONDS)
heartbeat('stopped', 'poll window ended')
print('worker stopped (poll window ended)')


## 終わったら

- Hermes 側は `lora_jobs.py fetch <job_id>` で成果物を受け取り、SHA-256 を照合します。
- 学習済み LoRA を Arli で使うには、Hugging Face へファイルを置き、プロンプトに
  `<lora:/huggingface.co/<repo>/<file>:0.8>` の形で指定します(HFへのアップロードは風児さんのトークンが要る別手順です)。
- Colab のランタイムは使い終わったら切断してください(無料枠の節約)。
